# Bài 8 · Xử lý dữ liệu thời gian

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn:

1. Đưa cột ngày về `datetime64`, khai thác `.dt.*` và `Timedelta`.
2. Làm việc trên trục thời gian: cắt lát chuỗi, `resample`, `rolling`.
3. So sánh đúng kiểu thời gian: kỳ trước (`pct_change`) vs **cùng kỳ** (`shift(12)`).
4. Né 3 bẫy: kỳ cụt, dd/mm vs mm/dd, ngày ghi nhận ≠ ngày sự kiện.

Dữ liệu: **690.112 review** của Santiago (bảng rút gọn 2 cột) + bảng listings.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rv = pd.read_csv(
    "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/reviews.csv",
    parse_dates=["date"],
)
print(rv.shape, rv.dtypes.tolist())
rv.head(3)

## 1. datetime và mỏ thông tin `.dt`

In [ ]:
# Bẫy quốc tịch trước tiên: 03/07 là ngày nào?
print(pd.to_datetime("03/07/2026"))                 # pandas mặc định: 7 tháng 3!
print(pd.to_datetime("03/07/2026", dayfirst=True))  # kiểu VN: 3 tháng 7

In [ ]:
# .dt.*: một cột datetime chứa sẵn hàng chục cột con
rv["date"].dt.year.value_counts().sort_index().tail(5)

In [ ]:
# Thứ mấy trong tuần nhiều review nhất? (0 = thứ Hai)
rv["date"].dt.dayofweek.value_counts().sort_index()

Chủ nhật (6) và thứ Hai (0) áp đảo — khách trả phòng cuối tuần rồi viết review.
**Ngày ghi nhận ≠ ngày sự kiện** — lần một.

In [ ]:
# Timedelta: "tuổi đời" của listing = snapshot - first_review
li = pd.read_csv(
    "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/data/listings.csv.gz",
    usecols=["id", "first_review", "number_of_reviews"],
    parse_dates=["first_review"],
)
tuoi_nam = (pd.Timestamp("2026-06-29") - li["first_review"]).dt.days / 365.25
tuoi_nam.describe().round(1)

## 2. Trục thời gian: cắt lát, resample, rolling

In [ ]:
r = rv.set_index("date").sort_index()

# Partial string indexing: cắt lát bằng chuỗi ngày
print("Năm 2026    :", len(r.loc["2026"]), "review")
print("Tháng 3/2026:", len(r.loc["2026-03"]), "review")

In [ ]:
# resample: groupby theo lịch (ME = tháng, tính đến cuối tháng)
theo_thang = r.resample("ME").size()
theo_thang.tail(4)

⚠️ **49 review "tháng 7"?** Snapshot chụp 29/06 — tháng 7 mới có vài giờ dữ liệu.
Đây là **kỳ cụt** (partial period): quên cắt là biểu đồ "lao dốc" giả tạo ngay.

In [ ]:
thang_du = theo_thang.iloc[:-1]        # nghi thức: bỏ kỳ chưa trọn

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(thang_du.index, thang_du.values, color="#999", lw=1, label="theo tháng")
ax.plot(thang_du.index, thang_du.rolling(6, center=True).mean(),
        color="#1E93AB", lw=2.5, label="trượt 6 tháng")
ax.legend(); ax.set_title("Review Santiago theo tháng — thấy gì giai đoạn 2020–2021?")
plt.tight_layout(); plt.show()

Cái hố 2020–2021 chính là **COVID-19** — không cần ai kể, dữ liệu tự nói.
`rolling` làm mượt nhiễu tháng-này-tháng-kia để xu hướng và "vết sẹo" lộ rõ.

## 3. So sánh theo thời gian

In [ ]:
# So kỳ liền trước: pct_change
print("So tháng trước (2 tháng cuối):")
print((thang_du.pct_change() * 100).round(1).tail(2))

# So CÙNG KỲ năm ngoái: shift(12)
yoy = (thang_du / thang_du.shift(12) - 1) * 100
print("\nSo cùng kỳ (2 tháng cuối):")
print(yoy.round(1).tail(2))

Tháng 6/2026: **−27%** so với tháng 5, nhưng **+7%** so với tháng 6/2025.
Dữ liệu có mùa vụ thì "so cùng kỳ" mới là phép so công bằng — mọi KPI theo thời gian
trong BTL nên báo cả hai.

In [ ]:
# Chỉ số mùa vụ: trung bình các năm gần đây, tháng nào cao/thấp?
gan_day = rv[(rv["date"] >= "2022-01-01") & (rv["date"] < "2026-07-01")]
mua_vu = gan_day.groupby(gan_day["date"].dt.month).size()
(mua_vu / mua_vu.mean() * 100).round(0)

Đỉnh T1–T4, đáy T9. Nhưng khoan — hè nam bán cầu là T12–T2, sao đỉnh lại dạt sang T3?

1. **Review trễ hơn chuyến đi** (viết sau khi trả phòng) — lần hai của "ngày ghi nhận ≠ ngày sự kiện".
2. **T2 chỉ có 28 ngày** — đếm theo tháng thiên vị tháng dài.

Đo được → mới giải thích được. Đoán trước → sai trước.

## 4. Bài tập tại lớp

### Bài 1 — Tháng vàng của từng năm

Với mỗi năm 2022–2025, tìm **tháng có nhiều review nhất** (gợi ý: `groupby` theo
`[dt.year, dt.month]` rồi `idxmax` theo từng năm, hoặc dùng `resample("ME")` + `groupby(index.year)`).

In [ ]:
# TODO Bài 1:
thang_2nam = thang_du.loc["2022":"2025"]
dinh = thang_2nam.groupby(thang_2nam.index.year).idxmax()
dinh

### Bài 2 — Tuần hay tháng?

Gộp lại theo **tuần** (`resample("W")`) cho giai đoạn 2025–2026, vẽ cùng một biểu đồ
với đường trượt 4 tuần. So với bản theo tháng: nhiễu hơn hay mịn hơn? Khi nào đáng
dùng tuần thay vì tháng?

In [ ]:
# TODO Bài 2:
theo_tuan = r.loc["2025":].resample("W").size().iloc[:-1]
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(theo_tuan.index, theo_tuan.values, color="#bbb", lw=0.8)
ax.plot(theo_tuan.index, theo_tuan.rolling(4, center=True).mean(), color="#E8890C", lw=2)
ax.set_title("Theo tuần: nhiều chi tiết hơn — và nhiều nhiễu hơn")
plt.tight_layout(); plt.show()

### Bài 3 — Listing "mới toanh" theo quý

Dùng bảng `li`: đếm số listing có `first_review` trong từng **quý** từ 2023 đến nay
(bỏ kỳ cụt cuối!). Quý nào thị trường Santiago "nở" mạnh nhất?

In [ ]:
# TODO Bài 3:
moi = li.dropna(subset=["first_review"]).set_index("first_review")
theo_quy = moi.loc["2023":].resample("QE").size().iloc[:-1]
print(theo_quy)
print("Quý nở mạnh nhất:", theo_quy.idxmax().date())

## 5. Thử thách về nhà 🏆 — Mùa vụ hai bán cầu

Lấy thêm reviews của **Rio de Janeiro**
(`https://data.insideairbnb.com/brazil/rj/rio-de-janeiro/2026-06-24/visualisations/reviews.csv`):

1. Tính chỉ số mùa vụ theo tháng (như mục 3) cho Rio giai đoạn 2022+ — nhớ cắt kỳ cụt.
2. Vẽ hai profile Santiago vs Rio trên cùng một hình (2 đường, 12 tháng).
3. Rio có "đỉnh lễ hội" tháng 2 (Carnival) không? Đỉnh của Rio lệch Santiago mấy tháng?
4. Viết 3 câu nhận xét — mỗi câu kèm con số.

*(Nhóm BTL nhận thành phố nam bán cầu: đây chính là một mục phân tích ăn điểm.)*

In [ ]:
RUN_CHALLENGE = False

if RUN_CHALLENGE:
    rio = pd.read_csv(
        "https://data.insideairbnb.com/brazil/rj/rio-de-janeiro/"
        "2026-06-24/visualisations/reviews.csv", parse_dates=["date"])
    ...

---

## Tóm tắt buổi học

| Ý chốt | Vì sao quan trọng |
|---|---|
| `parse_dates` + cẩn thận dd/mm; `.dt.*` là mỏ đặc trưng | Mọi phân tích thời gian bắt đầu ở đây |
| `resample` = groupby theo lịch; `rolling` làm mượt | Ngôn ngữ chuẩn của chuỗi thời gian |
| Kỳ cụt phải cắt trước khi vẽ/kết luận | Tránh "lao dốc" giả |
| Mùa vụ → so **cùng kỳ** (shift 12), không chỉ kỳ trước | KPI thời gian của BTL báo cả hai |
| Ngày ghi nhận ≠ ngày sự kiện; số review là proxy | Diễn giải trung thực — chuẩn minh bạch |

**Buổi sau (9):** thi giữa kỳ 🔒 — deck ôn tập đã có trên website môn học. Buổi 10 quay lại với
nghệ thuật làm sạch dữ liệu.